In [1]:
!pip install -U langchain langchain-openai requests


In [2]:
import os

os.environ["OPENROUTER_API_KEY"] = "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"
os.environ["OPENAI_API_KEY"] = os.environ["OPENROUTER_API_KEY"]
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"


In [3]:
import requests
from langchain.tools import tool

@tool
def fetch_url(url: str) -> str:
    """Fetch text content from a URL"""
    response = requests.get(url, timeout=10.0)
    response.raise_for_status()
    return response.text


In [4]:
from langchain.tools import tool
import json

@tool
def internet_search(query: str) -> str:
    """Mock search tool that returns 3 URLs."""
    urls = [
        "https://example.com/article1",
        "https://example.com/article2",
        "https://example.com/article3"
    ]
    return json.dumps(urls)


In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import JsonOutputParser

llm = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0.2,
)


In [7]:
import json

system_prompt = """
You are a research assistant.

Your job:
1. Rewrite the user's question into a short search query.
2. Call the tool: internet_search(query).
3. Take the top 3 URLs.
4. For each URL, call: fetch_url(url).
5. Read and synthesize the content.
6. Answer the user's question using ONLY that content.

Always respond in JSON with this structure:

{
  "search_query": "...",
  "urls": ["...", "...", "..."],
  "fetched": ["content1", "content2", "content3"],
  "final_answer": "..."
}
"""


In [8]:
def run_agent(question):
    # 1. اطلب من الـ LLM توليد search query
    msg = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ])


    try:
        data = json.loads(msg.content)
    except:
        return "LLM did not return valid JSON."


    urls = json.loads(internet_search.run(data["search_query"]))

    top3 = urls[:3]


    contents = []
    for url in top3:
        try:
            contents.append(fetch_url.run(url))
        except:
            contents.append("")


    summary = llm.invoke([
        SystemMessage(content="Summarize the following content and answer the user's question."),
        HumanMessage(content=f"User question: {question}\n\nContent:\n{contents}")
    ])

    return summary.content


In [9]:
result = run_agent("What are the benefits of browser automation using AI agents?")
print(result)


**Summary of the provided content:**  
The list you supplied contains three empty strings (`['', '', '']`), so there is no substantive text to summarize.

---

### Benefits of Browser Automation Using AI Agents

Even though the source material is empty, the general advantages of employing AI‑driven agents to automate web‑browser tasks are well documented. Below are the key benefits:

| Benefit | What It Means | Why It Matters |
|---------|---------------|----------------|
| **Increased Efficiency & Speed** | AI agents can perform repetitive actions (clicks, form fills, data extraction) far faster than a human. | Reduces the time needed for tasks like web scraping, testing, or data aggregation, enabling near‑real‑time processing of large volumes of web content. |
| **Higher Accuracy & Consistency** | AI models can follow precise instructions and handle variations in UI layout without fatigue. | Minimizes human error, ensuring reliable results across multiple runs and sites. |
| **Adapta